In [1]:
# ============================================================
# UPGRADED HYBRID INDOBERT + FEATURE EXTRACTION + TFIDF-SVD
# Low-Resource Indonesian Hate Speech Detection
# ============================================================

import os
import re
import random
import warnings
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import SequenceClassifierOutput

warnings.filterwarnings("ignore")

# ============================================================
# 1. GLOBALS
# ============================================================

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"
OUTPUT_DIR = "./paper_outputs_hybrid_plus"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")
PATH_ABUSIVE = os.path.join(DATA_DIR, "abusive.csv")

print("Using device:", DEVICE)

# ============================================================
# 2. SEED
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

# ============================================================
# 3. FILE LOADING
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception as e:
            last_error = e
    raise ValueError(f"Could not read file {path}. Last error: {last_error}")

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "not_hs", "0", "false", "no"}:
            return 0

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)

    return np.nan

def print_dataset_summary(df, name="dataset"):
    print("\n" + "=" * 70)
    print(f"DATASET: {name}")
    print("=" * 70)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    if "label" in df.columns:
        print("Label distribution:", Counter(df["label"].tolist()))

def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "IDHSD_713"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df

def load_572_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["comment_text", "tweet", "Tweet", "text", "Text"]
    possible_label_cols = ["class", "label", "Label", "HS"]

    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    label_col = next((c for c in possible_label_cols if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(f"Could not detect text/label columns in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "HS_572"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df

def load_re_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["Tweet", "tweet", "text", "Text"]
    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    if text_col is None:
        raise ValueError(f"No text column found in {path}")

    if "HS" in df.columns:
        label_col = "HS"
    elif "label" in df.columns:
        label_col = "label"
    else:
        raise ValueError(f"No HS/label column found in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "RE_13169"

    keep_cols = ["text", "label", "source"]
    if "Abusive" in df.columns:
        keep_cols.append("Abusive")

    df = df[keep_cols].dropna(subset=["text", "label"]).reset_index(drop=True)
    return df

def merge_datasets(datasets):
    merged = pd.concat(datasets, axis=0, ignore_index=True)
    merged["text"] = merged["text"].astype(str).str.strip()
    merged = merged[merged["text"].str.len() > 0].copy()
    merged = merged.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return merged

# ============================================================
# 4. TEXT NORMALIZATION
# ============================================================

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "wkwwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa",
    "xixi": "tertawa",
}

def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    if df.shape[1] < 2:
        raise ValueError("new_kamusalay.csv must have at least 2 columns.")
    col1, col2 = df.columns[:2]
    slang_dict = {}
    for _, row in df.iterrows():
        src = str(row[col1]).strip().lower()
        tgt = str(row[col2]).strip().lower()
        if src and src != "nan":
            slang_dict[src] = tgt
    return slang_dict

def load_abusive_lexicon(path):
    df = safe_read_csv(path)
    first_col = df.columns[0]
    return set(str(x).strip().lower() for x in df[first_col].dropna().tolist())

def reduce_repeated_chars(text):
    return REPEAT_CHAR_PATTERN.sub(r"\1\1", text)

def split_hashtag(match):
    return f" {match.group(1)} "

def normalize_special_tokens(text):
    tokens = text.split()
    out = []
    for tok in tokens:
        out.append(LAUGHTER_VARIANTS.get(tok, tok))
    return " ".join(out)

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\\n", " ")
    text = text.replace("\\/", "/")
    text = reduce_repeated_chars(text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(split_hashtag, text)
    text = text.lower()
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

def slang_normalize(text, slang_dict):
    tokens = text.split()
    return " ".join([slang_dict.get(tok, tok) for tok in tokens])

def preprocess_text(text, slang_dict=None):
    text = basic_clean_text(text)
    text = normalize_special_tokens(text)
    if slang_dict is not None:
        text = slang_normalize(text, slang_dict)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

# ============================================================
# 5. FEATURE LEXICONS
# ============================================================

IDENTITY_TARGET_TERMS = {
    "agama", "islam", "muslim", "kristen", "katolik", "hindu", "budha",
    "buddha", "konghucu", "cina", "tionghoa", "pribumi", "arab", "yahudi",
    "ulama", "ateis", "kafir", "nonmuslim", "non", "minoritas", "mayoritas"
}

RELIGIOUS_SLUR_TERMS = {
    "kafir", "murtad", "penista", "haram", "sesat", "laknat", "iblis"
}

ETHNIC_SLUR_TERMS = {
    "cina", "tionghoa", "aseng", "pribumi"
}

POLITICAL_INSULT_TERMS = {
    "cebong", "kampret", "buzzer", "antek", "koruptor", "pengkhianat"
}

NEGATION_TERMS = {
    "tidak", "bukan", "ga", "gak", "nggak", "tak"
}

INTENSIFIER_TERMS = {
    "banget", "sangat", "sekali", "parah", "terlalu", "amat"
}

THREAT_TERMS = {
    "bunuh", "usir", "hancur", "habisi", "bakar", "gantung", "serang"
}

POLITICAL_TOPIC_TERMS = {
    "pilkada", "debat", "gubernur", "presiden", "kampanye", "pemilu",
    "ahok", "anies", "jokowi", "sandi", "agus"
}

# ============================================================
# 6. HANDCRAFTED FEATURES
# ============================================================

def uppercase_ratio(raw_text):
    raw_text = str(raw_text)
    if len(raw_text) == 0:
        return 0.0
    n_upper = sum(1 for c in raw_text if c.isupper())
    n_alpha = sum(1 for c in raw_text if c.isalpha())
    if n_alpha == 0:
        return 0.0
    return n_upper / n_alpha

def repeated_char_count(raw_text):
    raw_text = str(raw_text)
    return len(REPEAT_CHAR_PATTERN.findall(raw_text.lower()))

def laughter_count(clean_text):
    return sum(1 for tok in clean_text.split() if tok == "tertawa")

def count_tokens_in_lexicon(clean_text, lexicon):
    return sum(1 for tok in clean_text.split() if tok in lexicon)

def elongated_token_count(raw_text):
    tokens = str(raw_text).split()
    return sum(1 for tok in tokens if REPEAT_CHAR_PATTERN.search(tok.lower()) is not None)

def all_caps_token_count(raw_text):
    tokens = str(raw_text).split()
    return sum(1 for tok in tokens if len(tok) > 1 and tok.isupper())

def punctuation_ratio(raw_text):
    raw_text = str(raw_text)
    if len(raw_text) == 0:
        return 0.0
    punct_count = sum(1 for c in raw_text if c in "!?.,;:")
    return punct_count / len(raw_text)

def build_handcrafted_features(df, abusive_lexicon):
    feats = pd.DataFrame(index=df.index)

    feats["abusive_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, abusive_lexicon)
    )
    feats["identity_target_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, IDENTITY_TARGET_TERMS)
    )
    feats["religious_slur_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, RELIGIOUS_SLUR_TERMS)
    )
    feats["ethnic_slur_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, ETHNIC_SLUR_TERMS)
    )
    feats["political_insult_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, POLITICAL_INSULT_TERMS)
    )
    feats["negation_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, NEGATION_TERMS)
    )
    feats["intensifier_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, INTENSIFIER_TERMS)
    )
    feats["threat_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, THREAT_TERMS)
    )
    feats["political_topic_count"] = df["clean_text"].apply(
        lambda txt: count_tokens_in_lexicon(txt, POLITICAL_TOPIC_TERMS)
    )

    feats["word_count"] = df["clean_text"].apply(lambda x: len(str(x).split()))
    feats["char_count"] = df["clean_text"].apply(lambda x: len(str(x)))
    feats["exclamation_count"] = df["text"].apply(lambda x: str(x).count("!"))
    feats["question_count"] = df["text"].apply(lambda x: str(x).count("?"))
    feats["uppercase_ratio"] = df["text"].apply(uppercase_ratio)
    feats["repeated_char_count"] = df["text"].apply(repeated_char_count)
    feats["elongated_token_count"] = df["text"].apply(elongated_token_count)
    feats["all_caps_token_count"] = df["text"].apply(all_caps_token_count)
    feats["punctuation_ratio"] = df["text"].apply(punctuation_ratio)
    feats["laughter_count"] = df["clean_text"].apply(laughter_count)

    return feats.astype(float)

# ============================================================
# 7. TF-IDF + SVD FUSION FEATURES
# ============================================================

def build_tfidf_svd_features(
    train_texts,
    val_texts,
    test_texts,
    unlabeled_texts=None,
    word_dim=64,
    char_dim=64
):
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=30000,
        sublinear_tf=True
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=20000,
        sublinear_tf=True
    )

    Xw_train = word_vec.fit_transform(train_texts)
    Xw_val = word_vec.transform(val_texts)
    Xw_test = word_vec.transform(test_texts)

    Xc_train = char_vec.fit_transform(train_texts)
    Xc_val = char_vec.transform(val_texts)
    Xc_test = char_vec.transform(test_texts)

    actual_word_dim = min(word_dim, Xw_train.shape[1] - 1) if Xw_train.shape[1] > 1 else 1
    actual_char_dim = min(char_dim, Xc_train.shape[1] - 1) if Xc_train.shape[1] > 1 else 1

    word_svd = TruncatedSVD(n_components=actual_word_dim, random_state=RANDOM_STATE)
    char_svd = TruncatedSVD(n_components=actual_char_dim, random_state=RANDOM_STATE)

    Xw_train_svd = word_svd.fit_transform(Xw_train)
    Xw_val_svd = word_svd.transform(Xw_val)
    Xw_test_svd = word_svd.transform(Xw_test)

    Xc_train_svd = char_svd.fit_transform(Xc_train)
    Xc_val_svd = char_svd.transform(Xc_val)
    Xc_test_svd = char_svd.transform(Xc_test)

    train_svd = np.concatenate([Xw_train_svd, Xc_train_svd], axis=1)
    val_svd = np.concatenate([Xw_val_svd, Xc_val_svd], axis=1)
    test_svd = np.concatenate([Xw_test_svd, Xc_test_svd], axis=1)

    unlabeled_svd = None
    if unlabeled_texts is not None:
        Xw_unlabeled = word_vec.transform(unlabeled_texts)
        Xc_unlabeled = char_vec.transform(unlabeled_texts)
        Xw_unlabeled_svd = word_svd.transform(Xw_unlabeled)
        Xc_unlabeled_svd = char_svd.transform(Xc_unlabeled)
        unlabeled_svd = np.concatenate([Xw_unlabeled_svd, Xc_unlabeled_svd], axis=1)

    return train_svd, val_svd, test_svd, unlabeled_svd

# ============================================================
# 8. PREPARE DATA
# ============================================================

def prepare_full_dataset():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)
    abusive_words = load_abusive_lexicon(PATH_ABUSIVE)

    ds_idhsd = load_idhsd_dataset(PATH_IDHSD)
    ds_572 = load_572_dataset(PATH_572)
    ds_re = load_re_dataset(PATH_RE)

    print_dataset_summary(ds_idhsd, "IDHSD")
    print_dataset_summary(ds_572, "572 dataset")
    print_dataset_summary(ds_re, "re_dataset")

    data = merge_datasets([ds_idhsd, ds_572, ds_re])
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].copy()

    print_dataset_summary(
        data[["clean_text", "label"]].rename(columns={"clean_text": "text"}),
        "Merged final dataset"
    )
    return data, abusive_words

def make_train_val_test_split(data, test_size=0.2, val_size=0.1):
    train_df, test_df = train_test_split(
        data,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=data["label"]
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def make_low_label_split(train_df, labeled_fraction=0.05):
    labeled_df, unlabeled_df = train_test_split(
        train_df,
        train_size=labeled_fraction,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )
    return labeled_df.reset_index(drop=True), unlabeled_df.reset_index(drop=True)

# ============================================================
# 9. DATASET WITH TOKENIZED TEXT + NUMERIC FEATURES
# ============================================================

class HybridDataset(Dataset):
    def __init__(self, texts, labels, numeric_features, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=False,
            max_length=max_length
        )
        self.labels = list(labels)
        self.numeric_features = np.asarray(numeric_features, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["numeric_features"] = torch.tensor(self.numeric_features[idx], dtype=torch.float32)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class HybridCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        numeric_features = torch.stack([f["numeric_features"] for f in features])
        labels = torch.stack([f["labels"] for f in features])

        text_features = []
        for f in features:
            d = {k: v for k, v in f.items() if k not in ["numeric_features", "labels"]}
            text_features.append(d)

        batch = self.tokenizer.pad(
            text_features,
            padding=True,
            return_tensors="pt"
        )
        batch["numeric_features"] = numeric_features
        batch["labels"] = labels
        return batch

def build_weighted_sampler(labels):
    class_counts = Counter(labels)
    weights = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = [weights[int(lbl)] for lbl in labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

def get_class_weights(labels):
    counts = Counter(labels)
    total = sum(counts.values())
    n_classes = len(counts)
    weights = {}
    for cls, cnt in counts.items():
        weights[int(cls)] = total / (n_classes * cnt)
    ordered = [weights.get(i, 1.0) for i in sorted(weights.keys())]
    return torch.tensor(ordered, dtype=torch.float)

# ============================================================
# 10. HYBRID MODEL
# ============================================================

class HybridIndoBERTClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_numeric_features,
        num_labels=2,
        numeric_hidden_dim=64,
        fusion_hidden_dim=256,
        dropout=0.2
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.numeric_net = nn.Sequential(
            nn.Linear(num_numeric_features, numeric_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + numeric_hidden_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, num_labels)
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        numeric_features=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None
        )

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0]

        numeric_repr = self.numeric_net(numeric_features)
        fused = torch.cat([pooled, numeric_repr], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None
        )

# ============================================================
# 11. CUSTOM TRAINER
# ============================================================

class WeightedHybridTrainer(Trainer):
    def __init__(self, class_weights=None, train_sampler=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.train_sampler_override = train_sampler

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            class_weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self.train_sampler_override if self.train_sampler_override is not None else None,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True
        )

def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else np.nan,
    }

# ============================================================
# 12. EVALUATION
# ============================================================

def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="model"):
    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_prob is not None:
        try:
            result["roc_auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            result["roc_auc"] = np.nan
    else:
        result["roc_auc"] = np.nan
    return result

def tune_threshold_from_probs(y_true, y_prob):
    best_thr = 0.5
    best_score = -1.0
    for thr in np.arange(0.20, 0.81, 0.02):
        pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, pred, average="macro")
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr, best_score

# ============================================================
# 13. FEATURE PREPARATION
# ============================================================

def build_full_numeric_features(train_df, val_df, test_df, unlabeled_df, abusive_words):
    train_hand = build_handcrafted_features(train_df, abusive_words)
    val_hand = build_handcrafted_features(val_df, abusive_words)
    test_hand = build_handcrafted_features(test_df, abusive_words)
    unlabeled_hand = build_handcrafted_features(unlabeled_df, abusive_words)

    train_svd, val_svd, test_svd, unlabeled_svd = build_tfidf_svd_features(
        train_df["clean_text"].tolist(),
        val_df["clean_text"].tolist(),
        test_df["clean_text"].tolist(),
        unlabeled_df["clean_text"].tolist() if len(unlabeled_df) > 0 else [],
        word_dim=64,
        char_dim=64
    )

    train_numeric = np.concatenate([train_hand.values, train_svd], axis=1)
    val_numeric = np.concatenate([val_hand.values, val_svd], axis=1)
    test_numeric = np.concatenate([test_hand.values, test_svd], axis=1)

    if len(unlabeled_df) > 0:
        unlabeled_numeric = np.concatenate([unlabeled_hand.values, unlabeled_svd], axis=1)
    else:
        unlabeled_numeric = np.empty((0, train_numeric.shape[1]))

    scaler = StandardScaler()
    train_numeric = scaler.fit_transform(train_numeric)
    val_numeric = scaler.transform(val_numeric)
    test_numeric = scaler.transform(test_numeric)
    if len(unlabeled_df) > 0:
        unlabeled_numeric = scaler.transform(unlabeled_numeric)

    return train_numeric, val_numeric, test_numeric, unlabeled_numeric, scaler

# ============================================================
# 14. TRAIN HYBRID MODEL
# ============================================================

def run_hybrid_indobert_model(
    model_checkpoint,
    train_df,
    val_df,
    test_df,
    train_numeric,
    val_numeric,
    test_numeric,
    run_name,
    max_length=128,
    epochs=50,
    batch_size=8,
    learning_rate=2e-5,
    seed=42
):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    train_dataset = HybridDataset(
        train_df["clean_text"].tolist(),
        train_df["label"].tolist(),
        train_numeric,
        tokenizer,
        max_length=max_length
    )
    val_dataset = HybridDataset(
        val_df["clean_text"].tolist(),
        val_df["label"].tolist(),
        val_numeric,
        tokenizer,
        max_length=max_length
    )
    test_dataset = HybridDataset(
        test_df["clean_text"].tolist(),
        test_df["label"].tolist(),
        test_numeric,
        tokenizer,
        max_length=max_length
    )

    num_numeric_features = train_numeric.shape[1]
    model = HybridIndoBERTClassifier(
        model_name=model_checkpoint,
        num_numeric_features=num_numeric_features,
        num_labels=2,
        numeric_hidden_dim=64,
        fusion_hidden_dim=256,
        dropout=0.2
    )

    class_weights = get_class_weights(train_df["label"].tolist())
    sampler = build_weighted_sampler(train_df["label"].tolist())
    collator = HybridCollator(tokenizer)

    run_output_dir = os.path.join(OUTPUT_DIR, run_name)
    os.makedirs(run_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=run_output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=seed,
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )

    trainer = WeightedHybridTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        compute_metrics=compute_metrics_for_trainer,
        class_weights=class_weights,
        train_sampler=sampler,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=50)],
    )

    trainer.train()

    val_output = trainer.predict(val_dataset)
    val_logits = val_output.predictions
    val_prob = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()
    best_thr, _ = tune_threshold_from_probs(val_df["label"], val_prob)
    val_pred = (val_prob >= best_thr).astype(int)

    test_output = trainer.predict(test_dataset)
    test_logits = test_output.predictions
    test_prob = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()
    test_pred = (test_prob >= best_thr).astype(int)

    print("\n" + "=" * 70)
    print(f"RESULTS: {run_name}")
    print("=" * 70)
    print(classification_report(test_df["label"], test_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(test_df["label"], test_pred))

    val_result = evaluate_predictions(val_df["label"], val_pred, val_prob, model_name=run_name)
    test_result = evaluate_predictions(test_df["label"], test_pred, test_prob, model_name=run_name)

    val_result.update({"split": "val", "best_threshold": best_thr, "seed": seed})
    test_result.update({"split": "test", "best_threshold": best_thr, "seed": seed})

    return pd.DataFrame([val_result, test_result]), trainer, tokenizer, best_thr

# ============================================================
# 15. OPTIONAL CONTROLLED SSL
# ============================================================

def predict_hybrid_probs(trainer, dataset):
    output = trainer.predict(dataset)
    logits = output.predictions
    return torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

def select_balanced_pseudo_labels(
    unlabeled_df,
    probs,
    round_idx=1,
    base_threshold=0.92,
    threshold_decay=0.05,
    max_add_per_round=1500
):
    current_thr = max(0.70, base_threshold - (round_idx - 1) * threshold_decay)

    preds = (probs >= 0.5).astype(int)
    confident = (probs >= current_thr) | (probs <= 1 - current_thr)

    cand = unlabeled_df.loc[confident].copy()
    if cand.empty:
        return pd.DataFrame(), unlabeled_df.copy(), current_thr

    cand["pseudo_label"] = preds[confident]
    cand["confidence"] = probs[confident]
    cand["confidence_margin"] = np.abs(cand["confidence"] - 0.5)

    per_class_cap = max_add_per_round // 2

    hs_cand = cand[cand["pseudo_label"] == 1].sort_values("confidence_margin", ascending=False).head(per_class_cap)
    non_cand = cand[cand["pseudo_label"] == 0].sort_values("confidence_margin", ascending=False).head(per_class_cap)

    selected = pd.concat([hs_cand, non_cand], ignore_index=True)

    if len(selected) < max_add_per_round:
        remaining = cand.drop(index=selected.index, errors="ignore").sort_values("confidence_margin", ascending=False)
        extra_needed = max_add_per_round - len(selected)
        selected = pd.concat([selected, remaining.head(extra_needed)], ignore_index=True)

    selected_texts = set(selected["clean_text"].tolist())
    remaining_df = unlabeled_df[~unlabeled_df["clean_text"].isin(selected_texts)].reset_index(drop=True)

    return selected.reset_index(drop=True), remaining_df, current_thr

def run_hybrid_ssl(
    labeled_df,
    unlabeled_df,
    val_df,
    test_df,
    abusive_words,
    scenario_name,
    max_rounds=2,
    seed=42
):
    current_labeled = labeled_df.copy().reset_index(drop=True)
    current_unlabeled = unlabeled_df.copy().reset_index(drop=True)
    frames = []

    for round_idx in range(1, max_rounds + 1):
        round_name = f"{scenario_name}_hybrid_ssl_round{round_idx}"

        train_num, val_num, test_num, unlabeled_num, scaler = build_full_numeric_features(
            current_labeled, val_df, test_df, current_unlabeled, abusive_words
        )

        hybrid_results, hybrid_trainer, hybrid_tokenizer, hybrid_thr = run_hybrid_indobert_model(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=current_labeled,
            val_df=val_df,
            test_df=test_df,
            train_numeric=train_num,
            val_numeric=val_num,
            test_numeric=test_num,
            run_name=f"hybrid_indobert_{round_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=seed
        )
        frames.append(hybrid_results)

        pred_dataset = HybridDataset(
            current_unlabeled["clean_text"].tolist(),
            [0] * len(current_unlabeled),
            unlabeled_num,
            hybrid_tokenizer,
            max_length=128
        )

        probs = predict_hybrid_probs(hybrid_trainer, pred_dataset)

        pseudo_df, current_unlabeled, used_thr = select_balanced_pseudo_labels(
            current_unlabeled,
            probs,
            round_idx=round_idx,
            base_threshold=0.92,
            threshold_decay=0.05,
            max_add_per_round=max(1000, int(len(current_labeled) * 2))
        )

        print("\n" + "#" * 70)
        print(f"HYBRID SSL ROUND: {round_name}")
        print("#" * 70)
        print("Used pseudo-label threshold:", used_thr)
        print("Pseudo-labeled added:", len(pseudo_df))
        print("Remaining unlabeled:", len(current_unlabeled))

        if pseudo_df.empty:
            break

        pseudo_add = pseudo_df.copy()
        if "label" in pseudo_add.columns:
            pseudo_add = pseudo_add.drop(columns=["label"])
        pseudo_add["label"] = pseudo_df["pseudo_label"].astype(int)
        pseudo_add = pseudo_add.reindex(columns=current_labeled.columns)

        current_labeled = pd.concat([current_labeled, pseudo_add], ignore_index=True)
        current_labeled = current_labeled.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ============================================================
# 16. MAIN PIPELINE
# ============================================================

def run_hybrid_pipeline_plus():
    data, abusive_words = prepare_full_dataset()
    train_df, val_df, test_df = make_train_val_test_split(data, test_size=0.2, val_size=0.1)

    scenarios = [0.05, 0.10, 0.20]
    all_results = []

    for frac in scenarios:
        scenario_name = f"{int(frac * 100)}pct"
        print("\n" + "=" * 80)
        print(f"LOW-LABEL SCENARIO: {scenario_name}")
        print("=" * 80)

        labeled_df, unlabeled_df = make_low_label_split(train_df, labeled_fraction=frac)

        train_num, val_num, test_num, unlabeled_num, scaler = build_full_numeric_features(
            labeled_df, val_df, test_df, unlabeled_df, abusive_words
        )

        hybrid_results, hybrid_trainer, hybrid_tokenizer, hybrid_thr = run_hybrid_indobert_model(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=labeled_df,
            val_df=val_df,
            test_df=test_df,
            train_numeric=train_num,
            val_numeric=val_num,
            test_numeric=test_num,
            run_name=f"hybrid_indobert_plus_{scenario_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=42
        )
        hybrid_results["label_fraction"] = frac
        all_results.append(hybrid_results)

        hybrid_ssl_results = run_hybrid_ssl(
            labeled_df=labeled_df,
            unlabeled_df=unlabeled_df,
            val_df=val_df,
            test_df=test_df,
            abusive_words=abusive_words,
            scenario_name=scenario_name,
            max_rounds=2,
            seed=42
        )
        if not hybrid_ssl_results.empty:
            hybrid_ssl_results["label_fraction"] = frac
            all_results.append(hybrid_ssl_results)

    final_results = pd.concat(all_results, axis=0, ignore_index=True)
    final_results.to_csv(os.path.join(OUTPUT_DIR, "hybrid_plus_results.csv"), index=False)

    print("\nTop TEST results by macro-F1:")
    print(
        final_results[final_results["split"] == "test"]
        .sort_values("macro_f1", ascending=False)
        [["label_fraction", "model", "accuracy", "macro_f1", "macro_precision", "macro_recall", "roc_auc", "best_threshold", "seed"]]
        .head(30)
        .to_string(index=False)
    )

    return final_results

# ============================================================
# 17. ENTRY
# ============================================================

if __name__ == "__main__":
    results = run_hybrid_pipeline_plus()

2026-04-19 12:28:42.889813: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda

DATASET: IDHSD
Shape: (713, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({0: 453, 1: 260})

DATASET: 572 dataset
Shape: (572, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({1: 286, 0: 286})

DATASET: re_dataset
Shape: (13169, 4)
Columns: ['text', 'label', 'source', 'Abusive']
Label distribution: Counter({0: 7608, 1: 5561})

DATASET: Merged final dataset
Shape: (14184, 2)
Columns: ['text', 'label']
Label distribution: Counter({0: 8251, 1: 5933})

LOW-LABEL SCENARIO: 5pct


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.685800,0.684123,0.506608,0.484509,0.580424,0.555104,0.619131
2,0.663600,0.668653,0.511894,0.467712,0.659044,0.574107,0.734946
3,0.601800,0.627389,0.626432,0.621543,0.686680,0.662560,0.777531
4,0.542600,0.564062,0.707489,0.707423,0.722265,0.724290,0.825691
5,0.417000,0.504262,0.761233,0.755621,0.754817,0.756635,0.846239
6,0.287000,0.504024,0.780617,0.770168,0.778473,0.766220,0.848539
7,0.178500,0.503539,0.777974,0.774882,0.773551,0.779585,0.862876
8,0.102900,0.551965,0.785022,0.780148,0.779108,0.781515,0.860998
9,0.064600,0.583233,0.785903,0.778314,0.781079,0.776372,0.864220
10,0.046300,0.632509,0.771806,0.768581,0.767276,0.773102,0.862201



RESULTS: hybrid_indobert_plus_5pct
              precision    recall  f1-score   support

           0     0.8219    0.8279    0.8249      1650
           1     0.7583    0.7506    0.7544      1187

    accuracy                         0.7956      2837
   macro avg     0.7901    0.7893    0.7897      2837
weighted avg     0.7953    0.7956    0.7954      2837

Confusion Matrix:
[[1366  284]
 [ 296  891]]


...


######################################################################
HYBRID SSL ROUND: 20pct_hybrid_ssl_round1
######################################################################
Used pseudo-label threshold: 0.92
Pseudo-labeled added: 4084
Remaining unlabeled: 4086


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.610600,0.537741,0.760352,0.758440,0.759190,0.766204,0.842593
2,0.265000,0.462681,0.803524,0.802225,0.803596,0.811882,0.893716
3,0.145600,0.485838,0.839648,0.829177,0.851341,0.821108,0.906083
4,0.096400,0.526662,0.832599,0.830738,0.829512,0.837767,0.916086
5,0.078500,0.570112,0.842291,0.838584,0.837497,0.839904,0.911362
6,0.059500,0.654131,0.813216,0.812440,0.816390,0.824641,0.910322
7,0.044700,0.635439,0.847577,0.843463,0.843363,0.843565,0.924912
8,0.034100,0.755928,0.826432,0.825769,0.830182,0.838660,0.923014
9,0.024300,0.779495,0.844053,0.841355,0.839328,0.845255,0.920603
10,0.017200,0.806710,0.840529,0.838539,0.836892,0.844880,0.917268



RESULTS: hybrid_indobert_20pct_hybrid_ssl_round2
              precision    recall  f1-score   support

           0     0.8849    0.8388    0.8612      1650
           1     0.7910    0.8484    0.8187      1187

    accuracy                         0.8428      2837
   macro avg     0.8380    0.8436    0.8400      2837
weighted avg     0.8456    0.8428    0.8434      2837

Confusion Matrix:
[[1384  266]
 [ 180 1007]]



######################################################################
HYBRID SSL ROUND: 20pct_hybrid_ssl_round2
######################################################################
Used pseudo-label threshold: 0.87
Pseudo-labeled added: 4078
Remaining unlabeled: 170

Top TEST results by macro-F1:
 label_fraction                                   model  accuracy  macro_f1  macro_precision  macro_recall  roc_auc  best_threshold  seed
           0.20 hybrid_indobert_20pct_hybrid_ssl_round2  0.842792  0.839966         0.837978      0.843573 0.918872            0.52    42
           0.20 hybrid_indobert_20pct_hybrid_ssl_round1  0.843497  0.838500         0.840207      0.837087 0.905993            0.44    42
           0.20              hybrid_indobert_plus_20pct  0.843497  0.838500         0.840207      0.837087 0.905993            0.44    42
           0.10 hybrid_indobert_10pct_hybrid_ssl_round2  0.836799  0.831654         0.833168      0.830383 0.905371            0.46    42
        